In [ ]:
import pandas as pd

In [ ]:
# ---------- 1) Download Census ACS 5-year county data (2023) ----------
# To pull socioeconomic variables from US Census API for county-level analysis
poverty_url = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1701_C03_001E&for=county:*" # Poverty status in the past 12 months below poverty level
income_url  = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1901_C01_012E&for=county:*" # Median household income in the past 12 months (in 2023 inflation-adjusted dollars)
edu_url     = "https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,S1501_C02_015E&for=county:*" # Educational attainment for the population 25 years and over: Bachelor's degree or higher

poverty = pd.read_json(poverty_url)
income  = pd.read_json(income_url)
edu     = pd.read_json(edu_url)

In [ ]:
# # Inspect raw Census API output: Census API returns the first row as column labels.
#   We verify the structure before cleaning.
print("Raw poverty head:")
print(poverty.head())

In [ ]:
# ---------- 2) Fix headers ----------
# Convert the first row into column headers so we can work with clean column names.
poverty.columns = poverty.iloc[0] # Set the first row as column headers
poverty = poverty[1:] # Remove the first row which is now the header
print(poverty.head(5)) # Verify the first few rows to confirm headers are set correctly

#do the same for income and edu
income.columns = income.iloc[0]
income = income[1:]

edu.columns = edu.iloc[0]
edu = edu[1:]

In [ ]:
# ---------- 3) Create county FIPS ----------
#   County FIPS = state (2 digits) + county (3 digits).
#   This will be the merge key for county-level fusion.
for df in [poverty, income, edu]:
    df["FIPS"] = df["state"] + df["county"]

print(poverty.head(5)) # Check the new FIPS column
print(income.head(5))
print(edu.head(5))

In [ ]:
# ---------- 4) Rename variables ----------
# Replace Census cryptic variable codes with meaningful names for readability and analysis.
poverty = poverty.rename(columns={"S1701_C03_001E": "poverty_rate"})
income  = income.rename(columns={"S1901_C01_012E": "median_income"})
edu     = edu.rename(columns={"S1501_C02_015E": "bachelor_plus_rate"})
print(poverty.head(5)) # Check renamed columns
print(income.head(5))
print(edu.head(5))

In [ ]:
# ---------- 5) Merge Census variables into one dataset ----------
# Combine poverty, income, and education variables into a single county-level dataframe.
census_df = poverty[["FIPS", "NAME", "poverty_rate"]].merge(
    income[["FIPS", "median_income"]], on="FIPS").merge(
        edu[["FIPS", "bachelor_plus_rate"]], on="FIPS")
print(census_df.head(5))

In [ ]:
# ---------- 6) Convert to numeric ----------
# ensure variables are usable for statistical analysis
# check data types
print(census_df.dtypes)

census_df["poverty_rate"] = pd.to_numeric(census_df["poverty_rate"], errors="coerce") # Convert poverty_rate to numeric, coercing errors to NaN
census_df["median_income"] = pd.to_numeric(census_df["median_income"], errors="coerce")
census_df["bachelor_plus_rate"] = pd.to_numeric(census_df["bachelor_plus_rate"], errors="coerce")
print(census_df.dtypes) # Check that the columns are now numeric

In [ ]:
# ---------- 7) Validate Census dataset ----------
# Confirm dataset shape and check missing values.
print("\nCensus dataset shape:", census_df.shape)
print(census_df.head())
print("\nMissing values check:")
print(census_df.isna().sum())

In [ ]:
# ---------- 8) Load NCES 2022–2023 district-level datasets ---------- (changed for Morgan's, refer to orignial if you uploading differently)
import os
import pandas as pd

base = r"C:\Users\YOUR FILE NAME!!!!!!!!!!!!\Downloads\NCES_2022-2023-20260410T174200Z-3-001\NCES_2022-2023"

lea_path     = os.path.join(base, "ccd_lea_029_2223_w_1a_083023.csv")
dropout_path = os.path.join(base, "CCD_LEA_032_2223_L_1a_PUB_050824.CSV")
grad_path    = os.path.join(base, "CCD_LEA_040_2223_L_1a_PUB_050824.CSV")

lea_df     = pd.read_csv(lea_path,     low_memory=False)
dropout_df = pd.read_csv(dropout_path, low_memory=False)
grad_df    = pd.read_csv(grad_path,    low_memory=False)

print("Shapes:", lea_df.shape, dropout_df.shape, grad_df.shape)

In [ ]:
# Validate NCES dataset sizes
# Check whether dataset sizes match expectation.
# LEA Universe file should be ~ 10k-20k rows (districts).
# Dropout and grad files may be larger due to subgroup breakdowns.
print("\nLEA dataset shape:", lea_df.shape)
print("Dropout dataset shape:", dropout_df.shape)
print("Graduate dataset shape:", grad_df.shape)

In [ ]:
# ---------- 9) Inspect column names----------
# Confirm that LEAID exists in all datasets as the district identifier.
print("LEA columns:\n", lea_df.columns.tolist())
print("Dropout columns:\n", dropout_df.columns.tolist())
print("Graduate columns:\n", grad_df.columns.tolist())

In [ ]:
# ---------- 10): Confirm common merge keys ----------
#   Identify shared columns and confirm LEAID exists as a unique district identifier.
common_cols = set(lea_df.columns).intersection(set(dropout_df.columns)).intersection(set(grad_df.columns))
print("\nCommon columns in all 3 datasets:", common_cols)

In [ ]:
# ---------- 11):Attempt a naive merge (for diagnostic purposes) ----------

naive_merge = lea_df.merge(dropout_df, on="LEAID", how="left")
naive_merge = naive_merge.merge(grad_df, on="LEAID", how="left")

print("\nNaive merged shape (should NOT be this large):", naive_merge.shape)

In [ ]:
# ---------- 11) Merge datasets (This step turns out explosion problem) ----------
# This step is used to demonstrate the join explosion problem when mergin datasets and will not be used in the final pipeline.

key = "LEAID" if "LEAID" in lea_df.columns else None # Check if LEAID is in the columns, otherwise set key to None
if key is None:
    raise ValueError("LEAID not found. Need to inspect column names.") # If LEAID is not found, raise an error to inspect column names

# Merge dropout + grad into LEA base
nces_df = lea_df.merge(dropout_df, on=key, how="left", suffixes=("", "_drop")) # Merge dropout data into LEA data, using left join to keep all LEAs, and adding suffixes to distinguish columns
nces_df = nces_df.merge(grad_df, on=key, how="left", suffixes=("", "_grad"))

print("\nMerged NCES shape:", nces_df.shape)
print(nces_df[[key]].head())

In [ ]:
# ---------- 12) : Verify subgroup breakdowns ----------
#   Confirm dropout and grad datasets contain multiple demographic categories per LEAID.
print("\nDropout TOTAL_INDICATOR distribution:")
print(dropout_df["TOTAL_INDICATOR"].value_counts().head(10))

print("\nGraduate TOTAL_INDICATOR distribution:")
print(grad_df["TOTAL_INDICATOR"].value_counts().head(10))

print("\nDropout breakdown examples:")
print(dropout_df[["GRADE", "RACE_ETHNICITY", "SEX"]].drop_duplicates().head(10))

print("\nGraduate breakdown examples:")
print(grad_df[["CREDENTIAL", "RACE_ETHNICITY", "SEX"]].drop_duplicates().head(10))

In [ ]:
# Conclusion:
#   The dropout and grad datasets are NOT one-row-per-district.
#   They contain subgroup-level rows (race, sex, grade, credential).
#   Therefore, we must aggregate them before merging with lea_df.

In [ ]:
# ---------- 13）: Extract district total dropout counts ----------
#   "Education Unit Total" indicates the district-level total (all subgroups combined).
#   This avoids double counting and provides a clean district-level measure.

dropout_total = dropout_df[
    dropout_df["TOTAL_INDICATOR"] == "Education Unit Total"
].copy()

print("\nDropout total rows shape:", dropout_total.shape)

In [2]:
# Aggregate dropout count per district (LEAID)
#   Even within total rows, some LEAIDs may appear multiple times (rare), so we sum as a safe method.
dropout_agg = dropout_total.groupby("LEAID")["STUDENT_COUNT"].sum().reset_index()
dropout_agg = dropout_agg.rename(columns={"STUDENT_COUNT": "dropout_count"})

print("Dropout aggregated shape:", dropout_agg.shape)
print(dropout_agg.head())

NameError: name 'dropout_total' is not defined

In [ ]:
# ---------- 14）: Extract district total graduate counts ----------
#   We also filter for "Education Unit Total" to ensure district-level totals.

grad_total = grad_df[
    grad_df["TOTAL_INDICATOR"] == "Education Unit Total"
].copy()

print("\nGraduate total rows shape:", grad_total.shape)

In [ ]:
# Aggregate graduate counts by district and credential type
#   This allows us to keep separate counts for diploma types if available.
grad_agg = grad_total.groupby(["LEAID", "CREDENTIAL"])["STUDENT_COUNT"].sum().reset_index()

print("Graduate aggregated shape:", grad_agg.shape)
print(grad_agg.head())

In [ ]:
# Pivot to wide format: each credential becomes a column
#   This makes the dataset easier to merge and use in ML models.
grad_pivot = grad_agg.pivot(
    index="LEAID",
    columns="CREDENTIAL",
    values="STUDENT_COUNT"
).reset_index()

print(grad_pivot.head())


In [ ]:
# Clean column names for modeling stability
#   Replace spaces with underscores and strip whitespace.
grad_pivot.columns = [str(c).strip().replace(" ", "_") for c in grad_pivot.columns]

print("Graduation pivot shape:", grad_pivot.shape)
print(grad_pivot.head())

In [ ]:
# ---------- 15): Merge aggregated dropout + graduation into district LEA table ----------
#   lea_df provides district metadata and identifiers.
#   dropout_agg and grad_pivot provide outcome variables.

nces_clean = lea_df.merge(dropout_agg, on="LEAID", how="left")
nces_clean = nces_clean.merge(grad_pivot, on="LEAID", how="left")

print("\nFinal cleaned NCES dataset shape:", nces_clean.shape)
print(nces_clean[["LEAID", "LEA_NAME", "FIPST", "dropout_count"]].head())

In [ ]:
# ---------- 16): Check if county code exists in NCES ----------
#   Determine whether we can directly fuse NCES with Census at the county level.

possible_county_cols = [c for c in lea_df.columns if "CON" in c.upper() or "COUNT" in c.upper()]
print("\nPossible county columns in lea_df:", possible_county_cols)

In [ ]:
# ---------- 17) Aggregate Census data to State level ----------
# Extract 2-digit state FIPS from the 5-digit county FIPS code
# e.g. "01001" → "01"

census_df["FIPS"] = census_df["FIPS"].astype(str).str.zfill(5)
census_df["state_fips"] = census_df["FIPS"].str[:2]

census_state = census_df.groupby("state_fips").agg(
    poverty_rate       = ("poverty_rate",      "mean"),
    median_income      = ("median_income",     "median"),
    bachelor_plus_rate = ("bachelor_plus_rate","mean")
).reset_index()

print("Census state-level shape:", census_state.shape)
print(census_state.head())

In [ ]:
# ---------- 18) Aggregate NCES data to State level ----------
# nces_clean already has FIPST (2-digit state FIPS as integer)
# Aggregate dropout count and each graduation credential column

nces_clean["state_fips"] = nces_clean["FIPST"].astype(str).str.zfill(2)

# Identify grad credential columns (everything added by the pivot)
grad_cols = [c for c in nces_clean.columns
             if c not in ["LEAID","LEA_NAME","FIPST","state_fips",
                          "STATENAME","ST","dropout_count"]]

agg_dict = {"dropout_count": "sum"}
for col in grad_cols:
    agg_dict[col] = "sum"

nces_state = nces_clean.groupby("state_fips").agg(agg_dict).reset_index()

print("NCES state-level shape:", nces_state.shape)
print(nces_state.head())

In [ ]:
# ---------- 19 FIXED) dropout_rate using credential columns ----------

# The grad pivot columns from cell 14 are the graduation counts
# Everything else in fused_df from lea_df is administrative (deleted most of fused) - exclude it

# Keep only the columns we actually need
analysis_cols = ["state_fips", "poverty_rate", "median_income",
                 "bachelor_plus_rate", "dropout_count"]

# Get only credential columns - these come from grad_pivot
# They are whatever columns grad_pivot added beyond LEAID
credential_cols = [c for c in grad_pivot.columns if c != "LEAID"]
print("Credential columns found:", credential_cols)

# Rebuild nces_state from scratch using only what matters
nces_clean["state_fips"] = nces_clean["FIPST"].astype(str).str.zfill(2)

agg_dict2 = {"dropout_count": "sum"}
for col in credential_cols:
    agg_dict2[col] = "sum"

nces_state2 = nces_clean.groupby("state_fips").agg(agg_dict2).reset_index()

# Merge with census state
fused2 = census_state.merge(nces_state2, on="state_fips", how="inner")

# Convert credential columns to numeric, clip negatives
fused2[credential_cols] = fused2[credential_cols].apply(
    pd.to_numeric, errors="coerce").clip(lower=0).fillna(0)

fused2["dropout_count"] = pd.to_numeric(
    fused2["dropout_count"], errors="coerce").clip(lower=0).fillna(0)

# Compute total grads and dropout rate
fused2["total_grads"] = fused2[credential_cols].sum(axis=1)
fused2["dropout_rate"] = (
    fused2["dropout_count"] /
    (fused2["dropout_count"] + fused2["total_grads"])
) * 100
fused2["dropout_rate"] = fused2["dropout_rate"].fillna(0)

print("\nFixed dataset shape:", fused2.shape)
print("\nSample:")
print(fused2[["state_fips","poverty_rate","median_income",
              "bachelor_plus_rate","dropout_rate"]].head(10).round(2))
print("\nDropout rate stats:")
print(fused2["dropout_rate"].describe().round(2))

In [ ]:
# ---------- 20 Rerun correlation and regression on clean data ----------
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("\nCorrelation with dropout_rate:")
print(fused2[["poverty_rate","median_income",
              "bachelor_plus_rate","dropout_rate"]].corr()["dropout_rate"].round(4))

X2 = fused2[["poverty_rate","median_income","bachelor_plus_rate"]].fillna(0)
y2 = fused2["dropout_rate"].fillna(0)

model2 = LinearRegression()
model2.fit(X2, y2)
y2_pred = model2.predict(X2)

print("\nR² Score:", round(r2_score(y2, y2_pred), 4))
print("Coefficients:")
for name, coef in zip(X2.columns, model2.coef_):
    print(f"  {name}: {round(coef, 4)}")
print("Intercept:", round(model2.intercept_, 4))

In [ ]:
# ---------- Correlation Heatmap ----------
import seaborn as sns
import matplotlib.pyplot as plt

corr_cols = ["poverty_rate", "median_income", "bachelor_plus_rate", "dropout_rate"]
corr_matrix = fused2[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation: Socioeconomic Indicators vs Dropout Rate (State Level)")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()
print("Saved: correlation_heatmap.png")

In [ ]:
# ---------- Scatter plots ----------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pairs = [
    ("poverty_rate",       "Poverty Rate (%)"),
    ("median_income",      "Median Income ($)"),
    ("bachelor_plus_rate", "Bachelor's Degree Rate (%)"),
]

for ax, (col, label) in zip(axes, pairs):
    ax.scatter(fused2[col], fused2["dropout_rate"], alpha=0.7, color="steelblue")
    ax.set_xlabel(label)
    ax.set_ylabel("Dropout Rate (%)")
    ax.set_title(f"{label} vs Dropout Rate")

plt.suptitle("State-Level Socioeconomic Factors vs Dropout Rate", y=1.02)
plt.tight_layout()
plt.savefig("scatter_plots.png", dpi=150)
plt.show()
print("Saved: scatter_plots.png")